In [1]:
from generate_utils import load_GraphModel, load_BiLSTMModel, load_TokenBiLSTMModel, load_LoRASEModel, load_AdapterModel
import torch
import numpy as np
import pickle
from GridMLM_tokenizers import CSGridMLMTokenizer
import os
from eval_utils import get_vecser_for_file, vecser_similarity_matrix
from dotenv import load_dotenv
from tqdm import tqdm
from eval_utils import vecser_similarity_evidence_for_files

from langchain_ollama import ChatOllama
from langchain.tools import tool

from ollama import chat
from ollama import ChatResponse

# Load environment variables from .env file
load_dotenv()

/home/maximos/miniconda3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# Initialize the ChatOllama model with the specified model name
model_name = 'qwen2.5-coder:7b'

# and initialize the ChatOllama instance
chat_model = ChatOllama(
    model=model_name,
    validate_model_on_init=True,
    temperature=0.7
)

In [3]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

def absoluteFilePaths(directory):
    file_names = []
    file_paths = []
    for dirpath,_,filenames in os.walk(directory):
        for f in filenames:
            if f.endswith( ('.mid', '.midi', '.mxl', '.xml', '.musicxml') ):
                file_names.append(f)
                file_paths.append(os.path.abspath(os.path.join(dirpath, f)))
    return file_names, file_paths
# end absoluteFilePaths

hook_file_names, hook_file_paths = absoluteFilePaths(os.getenv('VAL_HOOK'))
gjt_file_names, gjt_file_paths = absoluteFilePaths(os.getenv('VAL_GJT'))

device_name = 'cuda:2'
device = torch.device(device_name)

guide_arch = 'LoRA'
contra = True

adapter_model_path = f'saved_models/{guide_arch}/adapter/adapter_model_' + contra*'contra_' + 'jnhw.pt'
graph_adapter_model_path = f'saved_models/{guide_arch}/adapter/graph_model_' + contra*'contra_' + 'jnhw.pt'
token_adapter_model_path = f'saved_models/{guide_arch}/adapter/bilstm_model_' + contra*'contra_' + 'jnhw.pt'

token_adapter_model = load_TokenBiLSTMModel(token_adapter_model_path, tokenizer, device)
graph_adapter_model = load_GraphModel(graph_adapter_model_path, device)
adapter_model = load_AdapterModel(adapter_model_path, device)

token_adapter_model.eval()
graph_adapter_model.eval()
adapter_model.eval()

GuidanceAdapter(
  (proj): Linear(in_features=1024, out_features=512, bias=True)
)

In [17]:
f1 = gjt_file_paths[2]
f2 = gjt_file_paths[3]
print(f1)
print(f2)

bars_string, graph_res, token_res, adapter_res = vecser_similarity_evidence_for_files(
    f1,
    f2,
    tokenizer,
    graph_model=graph_adapter_model,
    bilstm_model=None,
    token_model=token_adapter_model,
    adapter_model=adapter_model,
    topk=10
)

/media/maindisk/data/mel_harm_CA_all/gjt_CA_test/From_this_moment_on.mxl
/media/maindisk/data/mel_harm_CA_all/gjt_CA_test/Bye_Bye_Baby.mxl


In [18]:
print(bars_string)

Piece 1:
bar 0: A:min6 
bar 1: A:min6 
bar 2: B:hdim7 
bar 3: E:7(b9) 
bar 4: D:min6 
bar 5: A:min6 
bar 6: G:min7 
bar 7: C:7 
bar 8: F:maj7 
bar 9: F:maj7 
bar 10: A#:7 
bar 11: A#:7 
bar 12: C:maj6 
bar 13: C:maj6 
bar 14: B:hdim7 
bar 15: E:7(b9) 

Piece 2:
bar 0: C:maj7 B:7 
bar 1: A#:9 A:7 
bar 2: D:min7 
bar 3: G:7 
bar 4: C:maj7 C:dim7 
bar 5: C:maj7 E:aug 
bar 6: F:maj6 
bar 7: G:7 
bar 8: C:maj7 E:7 
bar 9: A:min7 C:7 
bar 10: F:maj7 C#:dim7 
bar 11: D:min7 
bar 12: G:9 
bar 13: G:7 
bar 14: E:min7 A:7 
bar 15: D:min7 G:7 



In [19]:
print(graph_res)

Graph model evidence:
piece 1, bar 2: ['B:hdim7'] | piece 2, bar 12: ['G:9'] | 0.656131386756897
piece 1, bar 4: ['D:min6'] | piece 2, bar 12: ['G:9'] | 0.6552003026008606
piece 1, bar 12: ['C:maj6'] | piece 2, bar 9: ['A:min7', 'C:7'] | 0.6131623387336731
piece 1, bar 7: ['C:7'] | piece 2, bar 9: ['A:min7', 'C:7'] | 0.5724644660949707
piece 1, bar 9: ['F:maj7'] | piece 2, bar 10: ['F:maj7', 'C#:dim7'] | 0.5364347696304321
piece 1, bar 14: ['B:hdim7'] | piece 2, bar 6: ['F:maj6'] | 0.48680630326271057
piece 1, bar 2: ['B:hdim7'] | piece 2, bar 2: ['D:min7'] | 0.4866660237312317
piece 1, bar 11: ['A#:7'] | piece 2, bar 1: ['A#:9', 'A:7'] | 0.48593154549598694
piece 1, bar 4: ['D:min6'] | piece 2, bar 11: ['D:min7'] | 0.46473872661590576
piece 1, bar 4: ['D:min6'] | piece 2, bar 6: ['F:maj6'] | 0.4646536111831665



In [20]:
print(token_res)

Token model evidence:
piece 1, bar 7: ['C:7'] | piece 2, bar 9: ['A:min7', 'C:7'] | 0.8651601076126099
piece 1, bar 1: ['A:min6'] | piece 2, bar 6: ['F:maj6'] | 0.6960358023643494
piece 1, bar 9: ['F:maj7'] | piece 2, bar 2: ['D:min7'] | 0.6943466067314148
piece 1, bar 8: ['F:maj7'] | piece 2, bar 9: ['A:min7', 'C:7'] | 0.6375033855438232
piece 1, bar 4: ['D:min6'] | piece 2, bar 6: ['F:maj6'] | 0.6158115267753601
piece 1, bar 13: ['C:maj6'] | piece 2, bar 4: ['C:maj7', 'C:dim7'] | 0.5106632709503174
piece 1, bar 9: ['F:maj7'] | piece 2, bar 10: ['F:maj7', 'C#:dim7'] | 0.4683425724506378
piece 1, bar 12: ['C:maj6'] | piece 2, bar 0: ['C:maj7', 'B:7'] | 0.4192512631416321
piece 1, bar 13: ['C:maj6'] | piece 2, bar 5: ['C:maj7', 'E:aug'] | 0.4108799993991852
piece 1, bar 13: ['C:maj6'] | piece 2, bar 6: ['F:maj6'] | 0.4028232991695404



In [21]:
print(adapter_res)

Adapter model evidence:
piece 1, bar 7: ['C:7'] | piece 2, bar 9: ['A:min7', 'C:7'] | 0.6178734302520752
piece 1, bar 2: ['B:hdim7'] | piece 2, bar 12: ['G:9'] | 0.6033002734184265
piece 1, bar 4: ['D:min6'] | piece 2, bar 12: ['G:9'] | 0.5959640741348267
piece 1, bar 1: ['A:min6'] | piece 2, bar 4: ['C:maj7', 'C:dim7'] | 0.5892513990402222
piece 1, bar 8: ['F:maj7'] | piece 2, bar 10: ['F:maj7', 'C#:dim7'] | 0.5685641765594482
piece 1, bar 5: ['A:min6'] | piece 2, bar 0: ['C:maj7', 'B:7'] | 0.5628361701965332
piece 1, bar 10: ['A#:7'] | piece 2, bar 1: ['A#:9', 'A:7'] | 0.5270311236381531
piece 1, bar 5: ['A:min6'] | piece 2, bar 5: ['C:maj7', 'E:aug'] | 0.5265761017799377
piece 1, bar 12: ['C:maj6'] | piece 2, bar 9: ['A:min7', 'C:7'] | 0.5135286450386047
piece 1, bar 4: ['D:min6'] | piece 2, bar 6: ['F:maj6'] | 0.5130671262741089



In [22]:
central_prompt = '''
You are a music harmony expert and you will be give the chord sequences of two pieces, per bar.
You job is to provide an account of the similarities between the two harmonies.\n\n
'''

tool_prompt = '''
If it helps, you can use the output of a model that assessed the following similarities (maximum 1, minimum -1)
between bars of the two pieces:
'''

In [23]:
print(central_prompt + bars_string)


You are a music harmony expert and you will be give the chord sequences of two pieces, per bar.
You job is to provide an account of the similarities between the two harmonies.


Piece 1:
bar 0: A:min6 
bar 1: A:min6 
bar 2: B:hdim7 
bar 3: E:7(b9) 
bar 4: D:min6 
bar 5: A:min6 
bar 6: G:min7 
bar 7: C:7 
bar 8: F:maj7 
bar 9: F:maj7 
bar 10: A#:7 
bar 11: A#:7 
bar 12: C:maj6 
bar 13: C:maj6 
bar 14: B:hdim7 
bar 15: E:7(b9) 

Piece 2:
bar 0: C:maj7 B:7 
bar 1: A#:9 A:7 
bar 2: D:min7 
bar 3: G:7 
bar 4: C:maj7 C:dim7 
bar 5: C:maj7 E:aug 
bar 6: F:maj6 
bar 7: G:7 
bar 8: C:maj7 E:7 
bar 9: A:min7 C:7 
bar 10: F:maj7 C#:dim7 
bar 11: D:min7 
bar 12: G:9 
bar 13: G:7 
bar 14: E:min7 A:7 
bar 15: D:min7 G:7 



In [24]:
model_name = 'deepseek-r1:14b'
# model_name = 'qwen2.5-coder:14b'

In [25]:
response: ChatResponse = chat(
  model=model_name,
  messages=[
    {
      'role': 'user',
      'content': central_prompt + bars_string,
    }
  ],
  keep_alive=0
)
basic_response = response['message']['content']
print(basic_response)



The two musical pieces exhibit several similarities in their harmonic structures and chord usage, despite differing key centers and specific chord choices. Here's a detailed comparison:

1. **Chord Types and Extended Harmony:**
   - Both pieces employ extended harmony techniques, including half-diminished seventh (hdim7), dominant seventh with alterations (e.g., E7(b9)), and augmented chords.
   - They both use major sixth (maj6) and minor seventh (min7) chords, which contribute to a similar harmonic texture.

2. **Key Centers:**
   - While Piece 1 is rooted in A minor and related keys, Piece 2 centers around C major but explores other keys. Despite differing tonalities, both pieces use modulations, suggesting a shared approach to harmonic complexity.

3. **Specific Chord Matches:**
   - In bar 6 of each piece, F:maj6 appears in both (Piece1 bar6 and Piece2 bar6).
   - Dominant seventh chords are present in similar contexts: C:7 in Piece1 bar7 and G:7 in Piece2 bar7.

4. **Harmonic P

In [26]:
response: ChatResponse = chat(
  model=model_name,
  messages=[
    {
      'role': 'user',
      'content': central_prompt + bars_string + tool_prompt + token_res,
    }
  ],
  keep_alive=0
)
token_response = response['message']['content']
print(token_response)



**Analysis of Harmonic Similarities Between Two Pieces**

1. **Matching Chords and Functions:**
   - Both pieces utilize dominant 7th chords, such as C:7 in Piece 1 (bar 7) matching with C:7 in Piece 2 (bar 9), which is a strong similarity (score of 0.865). This suggests a common cadential pattern or harmonic function.

2. **Common Chord Types and Progressions:**
   - The use of minor chords progressing to dominant 7ths is evident in both pieces, creating tension and resolution. For example, A:min6 in Piece 1 leading to B:hdim7 mirrors similar progressions in Piece 2, contributing to harmonic movement.

3. **Key Relationships and Harmonic Movement:**
   - Despite starting in different keys (A minor for Piece 1 and C major for Piece 2), both pieces traverse through related keys like E and G. This common key usage facilitates similar harmonic transitions.

4. **Extended Chords and Mood:**
   - Both pieces employ extended chords (7ths, 9ths, diminished, augmented) which evoke comparable

In [27]:
response: ChatResponse = chat(
  model=model_name,
  messages=[
    {
      'role': 'user',
      'content': central_prompt + bars_string + tool_prompt + graph_res,
    }
  ],
  keep_alive=0
)
graph_response = response['message']['content']
print(graph_response)



**Analysis of Harmonic Similarities Between Two Pieces**

1. **Introduction and Structure:**
   - Both pieces are 16 bars long, indicating a comparable structure in length.

2. **Key Centers and Modulations:**
   - The harmonic journey in both pieces involves multiple key shifts, suggesting an exploration of various tonal centers.
   - Notable keys include A, B, C, D, E, F, G, with each piece modulating through these to create complexity.

3. **Common Chord Types and Functions:**
   - Both utilize extended chords such as seventh (7), half-diminished (hdim7), major seventh (maj7), minor sixth (min6), augmented (aug), and diminished (dim) chords.
   - The presence of similar chord functions, like dominant chords for tension and resolution, indicates a shared harmonic language.

4. **Specific Chord Comparisons:**
   - Piece 1's B:hdim7 (bar 2) aligns with Piece 2's G9 (bar 12), suggesting a functional similarity in creating tonal tension.
   - The use of F:maj7 in both pieces (Piece 1 b

In [28]:
response: ChatResponse = chat(
  model=model_name,
  messages=[
    {
      'role': 'user',
      'content': central_prompt + bars_string + tool_prompt + adapter_res,
    }
  ],
  keep_alive=0
)
adapter_response = response['message']['content']
print(adapter_response)



The two musical pieces exhibit several similarities in their harmonic structures, despite differing in specific chord choices and key centers. Here's a detailed breakdown of the similarities:

1. **Chord Types and Functions**:
   - Both pieces utilize dominant seventh chords (e.g., C:7 in piece 1 and G:7 in piece 2), which often serve similar functional roles in harmonic progressions, such as leading to a tonic chord.
   - They both employ minor sixth chords (min6) and major seventh chords (maj7), common in jazz harmonies, indicating a stylistic alignment.

2. **Progression Similarities**:
   - The sequences show related harmonic movements. For instance, piece 1's B:hdim7 progression leads to E:7(b9), while piece 2 features G:7 and C:maj7, suggesting similar functional steps in different keys.
   - The use of diminished and augmented chords (e.g., hdim7 and aug) points to comparable harmonic functions and effects.

3. **Key Modulations**:
   - Both pieces modulate through multiple ke